# Phase 7 - Generation (Retrieval-Augmented Query-Focused Summarisation)

The ablation that connects retrieval choice to answer quality. For each of the **60 frozen
golden queries** and each of the **three retrievers** {bm25, dense, hybrid}, retrieve the
**top-k = 5** context, assemble it under **one fixed prompt**, and call a **fixed generator**
(`gpt-4o-mini`, temperature 0, seed 42). The retriever is the only manipulated variable.

**Hard checkpoint:** this notebook GENERATES → PERSISTS → lets you INSPECT. It does **not**
evaluate. Phase 8 is built only after you have inspected the saved artifact here.

Order: assert golden hash → build 3 retrievers (dense from cache) → smoke-test one call →
generate 180 answers → **save + SHA-256** → write `phase_07_summary.json` → inspect.


<hr>

abstained is purely about the model's output text:

- abstained: true → the model emitted the exact sentinel sentence ("The provided context does not contain enough information to answer this question.") → it declined.
- abstained: false → the model wrote something else → it answered.

In [ ]:
import sys
from pathlib import Path

# Path.cwd().parent
sys.path.insert(0, str(Path.cwd().parent))

In [16]:
# %load_ext autoreload
# %autoreload 2

%reload_ext autoreload

import sys, json, time
from pathlib import Path

# Run from the repo root (where src/ and data/ live). Adjust if needed.
# ROOT = Path.cwd()
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
print("pandas", pd.__version__)

# --- canonical paths ---
CHUNKS_PATH  = ROOT / "data" / "processed" / "chunks_n200.parquet"
GOLDEN_PATH  = ROOT / "data" / "processed" / "golden_queries.parquet"
EMB_CACHE    = ROOT / "data" / "embeddings"
OUT_PATH     = ROOT / "data" / "processed" / "phase_07_generations.parquet"
SUMMARY_PATH = ROOT / "notes" / "phase_07_summary.json"
PREVIEW_PATH = ROOT / "notes" / "phase_07_preview.jsonl"

# --- the frozen golden-set hash (source of truth: thesis_context_summary.md) ---
GOLDEN_SHA256 = "325634f065147a49d6fba6e0fb021107d536b1aa717bcbbb4a10b68b93c0e72e"
K = 5

pandas 3.0.2


## 1. Load corpus + golden set, assert the golden hash (HARD GATE)

In [17]:
from src.retrieval.bm25_retriever import load_chunks
from src.retrieval.golden_dataset_builder import load_golden_set, hash_golden_set

chunks_df = load_chunks(CHUNKS_PATH)
golden_df = load_golden_set(GOLDEN_PATH)

actual_hash = hash_golden_set(golden_df)
assert actual_hash == GOLDEN_SHA256, (
    f"GOLDEN HASH MISMATCH\n expected {GOLDEN_SHA256}\n got      {actual_hash}\n"
    "Refusing to run: this would not be the same eval set as Phases 4/5/6."
)
print("golden hash assert: PASS")
print(f"chunks: {len(chunks_df):,}   golden queries: {len(golden_df)}")
print(golden_df['source'].value_counts().to_dict())

golden hash assert: PASS
chunks: 21,050   golden queries: 60
{'earnings': 30, 'edgar': 30}


## 2. Build the three retrievers (dense loads from the Phase-5 cache)

The dense embeddings are cached under `data/embeddings/` keyed by model + chunk config +
chunk-id fingerprint. Because the corpus is the identical `chunks_n200.parquet`, this is a
**cache hit** (~tens of seconds), not a 33-minute re-embed. If the cache is missing it will
re-embed; that is the only slow path.

In [18]:
from src.retrieval.bm25_retriever import BM25Retriever
from src.retrieval.dense_retriever import DenseRetriever
from src.retrieval.hybrid_retriever import HybridRetriever

t0 = time.time()
bm25 = BM25Retriever(chunks_df)
print(f"bm25 built in {time.time()-t0:.1f}s  (n={len(bm25)})")

t0 = time.time()
dense = DenseRetriever.from_model(chunks_df, cache_dir=str(EMB_CACHE), use_cache=True)
print(f"dense built in {time.time()-t0:.1f}s  (backend={dense.backend})")

hybrid = HybridRetriever(bm25, dense, k=60, candidate_depth=100)
print(f"hybrid composed  (n={len(hybrid)})")

retrievers = {"bm25": bm25, "dense": dense, "hybrid": hybrid}

bm25 built in 1.9s  (n=21050)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

dense built in 55.3s  (backend=faiss)
hybrid composed  (n=21050)


## 3. Wire the fixed generator (gpt-4o-mini, temperature 0, seed 42)

In [19]:
from openai import OpenAI
from dotenv import load_dotenv
from src.generation.generator import (
    make_openai_complete_fn, GENERATION_SYSTEM_PROMPT, PROMPT_VERSION,
    prompt_fingerprint, DEFAULT_GEN_MODEL,
)

load_dotenv(override=True)
client = OpenAI()
complete_fn = make_openai_complete_fn(client, model=DEFAULT_GEN_MODEL,
                                      temperature=0.0, seed=42)

print("generator model:", DEFAULT_GEN_MODEL)
print("prompt_version :", PROMPT_VERSION, "| fingerprint:", prompt_fingerprint())
print("--- fixed system prompt (the control) ---")
print(GENERATION_SYSTEM_PROMPT)

generator model: gpt-4o-mini
prompt_version : p7_qfs_v1 | fingerprint: 9d051ed099c26666
--- fixed system prompt (the control) ---
You are a financial analyst assistant. Answer the user's question using ONLY the numbered context passages provided.
- Ground every statement in the passages; do not use any outside knowledge.
- Cite the passage number(s) you used in square brackets immediately after each claim, e.g. [1] or [2][3].
- Be specific: reproduce the exact figures, dates, and named terms from the passages when they are relevant to the question.
- If the passages do not contain enough information to answer the question, reply with exactly this sentence and nothing else: "The provided context does not contain enough information to answer this question."
- Keep the answer concise (a few sentences).


## 4. Smoke test - ONE generation before spending 180 calls

Guardrail: verifies the API key, model access and the whole assembly path on a single
query before the full sweep. Inspect the printed record; if it looks right, proceed.

In [20]:
from src.generation.generator import generate_one

q0 = golden_df.iloc[0]
rec0 = generate_one(
    query_text=q0["query_text"], relevant_ids=list(q0["relevant_chunk_ids"]),
    retriever=bm25, retriever_name="bm25", complete_fn=complete_fn,
    query_id=q0["query_id"], source=q0["source"], subtype=q0["subtype"], k=K,
)
print("query   :", rec0["query_text"])
print("gold in context:", rec0["gold_in_context"], "| gold_rank:", rec0["gold_rank"])
print("abstained:", rec0["abstained"], "| system_fingerprint:", rec0["system_fingerprint"])
print("answer  :", rec0["answer"])

query   : In the closing remarks of the call, what did the CEO thank participants for and say about the company's future?
gold in context: False | gold_rank: None
abstained: False | system_fingerprint: fp_7fa0fbf23d
answer  : In the closing remarks of the call, CEO Rick Smith thanked participants for their time and mentioned that "2016 is off to a solid beginning." He expressed the company's focus on ensuring the remainder of the year is successful with their "Axon market domination and our international expansion." He also looked forward to discussing the company's long-term trajectory in greater detail at the upcoming Analyst Day in New York [1].


## 5. Run the full 60 × 3 = 180 matrix

Cheap (well under \$1 at gpt-4o-mini). Deterministic row order; the saved hash is
order-independent regardless.

In [21]:
from src.generation.generator import run_generation_matrix

def _progress(done, total):
    if done == 1 or done % 30 == 0 or done == total:
        print(f"  {done:3d}/{total}")

t0 = time.time()
gen_df = run_generation_matrix(golden_df, retrievers, complete_fn, k=K, progress=_progress)
print(f"\n180 generations in {time.time()-t0:.1f}s   rows={len(gen_df)}")
assert len(gen_df) == len(golden_df) * len(retrievers) == 180, "expected 180 rows"
assert gen_df["prompt_fingerprint"].nunique() == 1, "prompt must be constant across all calls"
print("prompt constant across all calls: PASS")

    1/180
   30/180
   60/180
   90/180
  120/180
  150/180
  180/180

180 generations in 287.0s   rows=180
prompt constant across all calls: PASS


## 6. Persist + SHA-256 the frozen artifact

In [22]:
from src.generation.generator import save_generations

gen_hash = save_generations(gen_df, OUT_PATH)
print("saved   :", OUT_PATH)
print("sha256  :", gen_hash)

saved   : d:\General IT\AI-ML-LJMU\final_thesis\code\data\processed\phase_07_generations.parquet
sha256  : 862cf6931f3b8fd88c920a7c98c53cda5412c9617f5472cafefe958fcada5ab1


## 7. Inspection summary (gold-in-context + abstention by retriever × source)

In [23]:
# gold_in_context rate — the mediating variable. Should track Recall@5:
#   bm25 ~0.78, hybrid ~0.82, dense ~0.67 overall.
gic = (gen_df.groupby("retriever")["gold_in_context"].mean()
       .rename("gold_in_context_rate").to_frame())
print("gold_in_context rate by retriever:"); print(gic.round(3)); print()

gic_src = (gen_df.groupby(["retriever", "source"])["gold_in_context"]
           .mean().rename("rate").reset_index())
print("gold_in_context rate by retriever × source:")
print(gic_src.pivot(index="retriever", columns="source", values="rate").round(3)); print()

abst = (gen_df.groupby("retriever")["abstained"].mean()
        .rename("abstention_rate").to_frame())
print("abstention rate by retriever:"); print(abst.round(3))

gold_in_context rate by retriever:
           gold_in_context_rate
retriever                      
bm25                      0.783
dense                     0.667
hybrid                    0.817

gold_in_context rate by retriever × source:
source     earnings  edgar
retriever                 
bm25          0.833  0.733
dense         0.800  0.533
hybrid        0.933  0.700

abstention rate by retriever:
           abstention_rate
retriever                 
bm25                 0.067
dense                0.050
hybrid               0.050


In [24]:
# Eyeball a few full records, including a dense gold-MISSING case if one exists.
import textwrap
def show(r):
    print("="*70)
    print(f"{r['query_id']} | {r['retriever']} | {r['source']}/{r['subtype']} | "
          f"gold_in_context={r['gold_in_context']} rank={r['gold_rank']} abstained={r['abstained']}")
    print("Q:", r["query_text"])
    print("A:", textwrap.fill(str(r["answer"]), 100))

show(gen_df.iloc[0])
miss = gen_df[(gen_df.retriever=="dense") & (~gen_df.gold_in_context)]
if len(miss):
    print("\n--- a dense gold-MISSING case (expect abstention or ungrounded answer) ---")
    show(miss.iloc[0])

q000 | bm25 | earnings/qa | gold_in_context=False rank=nan abstained=False
Q: In the closing remarks of the call, what did the CEO thank participants for and say about the company's future?
A: In the closing remarks of the call, CEO Rick Smith thanked participants for their time and mentioned
that "2016 is off to a solid beginning." He expressed the company's focus on ensuring the remainder
of the year is successful with their "Axon market domination and our international expansion." He
also indicated that they look forward to discussing the long-term trajectory of the business in
greater detail at their upcoming Analyst Day in New York [1].

--- a dense gold-MISSING case (expect abstention or ungrounded answer) ---
q003 | dense | earnings/prepared_remarks | gold_in_context=False rank=nan abstained=False
Q: What was the year-on-year growth rate of networking revenue, and what dollar figure did it reach?
A: The year-on-year growth rate of networking revenue was 16%, increasing from $281

## 8. Write `phase_07_summary.json` (real numbers) + a preview for inspection

In [25]:
def _rate_by(group_cols, col):
    g = gen_df.groupby(group_cols)[col].mean()
    return {("/".join(map(str, k)) if isinstance(k, tuple) else str(k)): round(float(v), 4)
            for k, v in g.items()}

summary = {
    "phase": 7,
    "task": "retrieval_augmented_query_focused_summarisation",
    "generator": {"model": DEFAULT_GEN_MODEL, "temperature": 0.0, "seed": 42,
                  "k": K, "prompt_version": PROMPT_VERSION,
                  "prompt_fingerprint": prompt_fingerprint()},
    "retrievers": sorted(retrievers.keys()),
    "n_generations": int(len(gen_df)),
    "n_queries": int(len(golden_df)),
    "golden_set": {"path": str(GOLDEN_PATH), "sha256": GOLDEN_SHA256, "hash_assert": "PASS"},
    "generations_artifact": {"path": str(OUT_PATH), "sha256": gen_hash},
    "system_fingerprints_seen": sorted(
        {str(x) for x in gen_df["system_fingerprint"].dropna().unique()}),
    "gold_in_context_rate": {
        "overall": _rate_by("retriever", "gold_in_context"),
        "by_source": _rate_by(["retriever", "source"], "gold_in_context"),
    },
    "abstention_rate": {
        "overall": _rate_by("retriever", "abstained"),
        "by_source": _rate_by(["retriever", "source"], "abstained"),
    },
}
SUMMARY_PATH.parent.mkdir(parents=True, exist_ok=True)
SUMMARY_PATH.write_text(json.dumps(summary, indent=2))
print("wrote", SUMMARY_PATH)
print(json.dumps(summary, indent=2))

wrote d:\General IT\AI-ML-LJMU\final_thesis\code\notes\phase_07_summary.json
{
  "phase": 7,
  "task": "retrieval_augmented_query_focused_summarisation",
  "generator": {
    "model": "gpt-4o-mini",
    "temperature": 0.0,
    "seed": 42,
    "k": 5,
    "prompt_version": "p7_qfs_v1",
    "prompt_fingerprint": "9d051ed099c26666"
  },
  "retrievers": [
    "bm25",
    "dense",
    "hybrid"
  ],
  "n_generations": 180,
  "n_queries": 60,
  "golden_set": {
    "path": "d:\\General IT\\AI-ML-LJMU\\final_thesis\\code\\data\\processed\\golden_queries.parquet",
    "sha256": "325634f065147a49d6fba6e0fb021107d536b1aa717bcbbb4a10b68b93c0e72e",
    "hash_assert": "PASS"
  },
  "generations_artifact": {
    "path": "d:\\General IT\\AI-ML-LJMU\\final_thesis\\code\\data\\processed\\phase_07_generations.parquet",
    "sha256": "862cf6931f3b8fd88c920a7c98c53cda5412c9617f5472cafefe958fcada5ab1"
  },
  "system_fingerprints_seen": [
    "fp_31455a11c2",
    "fp_7fa0fbf23d",
    "fp_8324b7be19"
  ],
  "g

In [26]:
# Human-readable preview (NOT the canonical artifact — the parquet is). Easy to eyeball.
preview_cols = ["query_id","retriever","source","subtype","gold_in_context",
                "gold_rank","abstained","query_text","answer"]
with open(PREVIEW_PATH, "w", encoding="utf-8") as f:
    for _, r in gen_df[preview_cols].iterrows():
        f.write(json.dumps({c: (None if pd.isna(r[c]) else r[c]) for c in preview_cols},
                           ensure_ascii=False, default=str) + "\n")
print("wrote", PREVIEW_PATH, "(", len(gen_df), "lines )")

wrote d:\General IT\AI-ML-LJMU\final_thesis\code\notes\phase_07_preview.jsonl ( 180 lines )


## Figure: query disposition by retriever ---

In [30]:
gen_df.head(2)

,query_id,retriever,source,subtype,query_text,k,relevant_chunk_ids,retrieved_chunk_ids,retrieved_scores,gold_in_context,...,n_context,context_text,answer,abstained,gen_model,temperature,seed,system_fingerprint,prompt_version,prompt_fingerprint
0,q000,bm25,earnings,qa,"In the closing remarks of the call, what did t...",5,[earnings_0000_qa_chunk_030],"[earnings_0066_qa_chunk_014, earnings_0023_qa_...","[62.387834261144974, 61.41143764120822, 60.595...",False,...,5,[1]\nRick Smith : They are an enormous agency ...,"In the closing remarks of the call, CEO Rick S...",False,gpt-4o-mini-2024-07-18,0.0,42,fp_7fa0fbf23d,p7_qfs_v1,9d051ed099c26666
1,q001,bm25,earnings,qa,What productivity contribution percentage was ...,5,[earnings_0002_qa_chunk_012],"[earnings_0002_qa_chunk_007, edgar_0074_sectio...","[43.70750371948543, 38.19275338158239, 37.4804...",True,...,5,"[1]\nMike Hsu : Yes. So a few things, Robert. ...",The expected productivity contribution percent...,False,gpt-4o-mini-2024-07-18,0.0,42,fp_7fa0fbf23d,p7_qfs_v1,9d051ed099c26666


In [28]:
# --- Phase 7 figure: query disposition by retriever ---

FIG_DIR = ROOT / "reports" / "figures"

import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt, numpy as np
order = ["bm25", "dense", "hybrid"]
pa, ma, ab = [], [], []
for r in order:
    g = gen_df[gen_df.retriever == r]
    pa.append(int(((g.gold_in_context) & (~g.abstained)).sum()))
    ma.append(int(((~g.gold_in_context) & (~g.abstained)).sum()))
    ab.append(int(g.abstained.sum()))
pa, ma, ab = map(np.array, (pa, ma, ab))
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(order, pa, color="#55A868", label="gold in context -> answered")
ax.bar(order, ma, bottom=pa, color="#DD8452", label="gold MISSING -> answered (confabulation risk)")
ax.bar(order, ab, bottom=pa+ma, color="#B0B0B0", label="abstained")
for i in range(3):
    ax.text(i, pa[i]/2, pa[i], ha="center", va="center", color="white", fontweight="bold")
    ax.text(i, pa[i]+ma[i]/2, ma[i], ha="center", va="center", color="white", fontweight="bold")
    ax.text(i, pa[i]+ma[i]+ab[i]/2, ab[i], ha="center", va="center", fontsize=9)
ax.set_ylabel("queries (of 60)"); ax.set_ylim(0, 64)
ax.set_title("Phase 7 - query disposition by retriever (k=5, n=60)")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.10), ncol=1, frameon=False, fontsize=9)
plt.tight_layout(); plt.savefig(FIG_DIR / "phase07_query_disposition.png", dpi=150, bbox_inches="tight"); plt.show()

C:\Users\anshu\AppData\Local\Temp\ipykernel_57380\2649068254.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(FIG_DIR / "phase07_query_disposition.png", dpi=150, bbox_inches="tight"); plt.show()


### Inspection results

- "gold_in_context tracks Recall@5" — Does the right chunk reach the context as often as retrieval said it would? PASS, exactly (table below). Wiring confirmed.

| Retriever | gold present & answered | gold MISSING & answered | abstained | total |
|-----------|-------------------------|--------------------------|-----------|-------|
| bm25      | 46                      | 10                       | 4         | 60    |
| dense     | 39                      | 18                       | 3         | 60    |
| hybrid    | 47                      | 10                       | 3         | 60    |

- "Answers grounded / cite [n], dense gold-missing mostly abstain" — answers are citing [n] and read as grounded (q000, q003 both cite and are specific). The "mostly abstain" half did not hold — as explained, the model answers instead. That's not a failure; it's the finding that motivates the faithfulness judge. So: half confirmed-as-expected, half informatively-different. No action.

- "system_fingerprints_seen" — documented. Fine.

      "system_fingerprints_seen": [
         "fp_31455a11c2",
         "fp_7fa0fbf23d",
         "fp_8324b7be19"
      ]

- "SHA-256 recorded" — yes: 862cf693…5ab1 is in the summary JSON. Phase 8 will assert it.